# 한국어 TTS 벤치마크 — Google Colab (무료 GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/techgit01/tts-bmt-poc/blob/main/notebooks/tts_bmt_colab.ipynb)

4종 중 **추론형 2종**(Supertonic, piper-plus)은 pip 설치만으로 바로 합성되고,
**학습형 2종**(kr-custom-tts, SCE-TTS)은 본인 음성으로 **학습**해야 모델 산출물이 나옵니다.
학습형은 GPU가 필요하므로 이 노트북을 무료 Colab에서 돌립니다.

**흐름:** GPU 확인 → 각 엔진 설치/학습 → 모델 산출물(`.onnx`/`.pt`) 생성 → 다운로드 →
로컬 저장소의 `src/tts_bmt/engines.py` 에서 각 엔진 `model_path` 로 연결 → `./run.sh` 로 실측.

> 학습 세부 명령은 업스트림마다 다르므로 **각 섹션의 공식 가이드 링크**를 따르세요.
> 아래 셀은 검증 가능한 공통 단계(설치/클론/내보내기)와 `# TODO` 표시로 구성된 스캐폴드입니다.

| 엔진 | 방식 | 업스트림 |
|---|---|---|
| Supertonic 3 | pip 추론 | https://github.com/supertone-inc/supertonic |
| piper-plus | pip 추론 + 음성모델 | https://github.com/ayutaz/piper-plus |
| kr-custom-tts | ESPnet 학습 | https://github.com/seastar105/kr-custom-tts |
| SCE-TTS | Tacotron2 학습 | https://gist.github.com/yunho0130 |

## 0. 환경 확인 (GPU / 런타임)
Colab 메뉴: **런타임 > 런타임 유형 변경 > 하드웨어 가속기: GPU (T4)** 선택 후 실행.

In [ ]:
!nvidia-smi || echo '⚠️ GPU 미할당 — 런타임 유형을 GPU로 변경하세요 (학습형 2종에 필요)'
import sys; print('Python', sys.version)

## 1. (선택) 이 저장소 클론
벤치마크 하네스(`tts-bmt`)와 `docs/` 산출물 구조를 Colab에서도 쓰려면 클론합니다.

In [ ]:
%cd /content
![ -d tts-bmt-poc ] || git clone https://github.com/techgit01/tts-bmt-poc.git
%cd /content/tts-bmt-poc
!pip -q install -e . && python -c "import tts_bmt; print('tts-bmt OK')"

## 2. Supertonic 3 (추론형 — 설치만)
온디바이스 ONNX. 학습 불필요. 가이드: https://github.com/supertone-inc/supertonic

In [ ]:
!pip -q install supertonic
# TODO: 업스트림 README 의 추론 API 에 맞춰 호출 (예시 — 실제 함수명은 README 확인)
#   from supertonic import Supertonic
#   tts = Supertonic()
#   tts.tts_to_file('안녕하세요, 고객님.', '/content/supertonic_s1.wav', lang='ko')
print('Supertonic 설치 완료 — 추론 호출은 업스트림 README 참고')

## 3. piper-plus (추론형 — 설치 + 한국어 음성모델)
VITS. 한국어 `.onnx` 음성모델이 별도로 필요합니다. 가이드: https://github.com/ayutaz/piper-plus

In [ ]:
!pip -q install piper-plus
# TODO: 업스트림에서 한국어 음성모델(.onnx + .onnx.json) 경로/다운로드를 확인해 받기
#   !wget -O /content/ko_voice.onnx       <업스트림에서 제공하는 URL>
#   !wget -O /content/ko_voice.onnx.json  <업스트림에서 제공하는 URL>
# 추론 호출 예시(실제 API 는 README 확인):
#   from piper_plus import PiperVoice
#   voice = PiperVoice.load('/content/ko_voice.onnx')
print('piper-plus 설치 완료 — 한국어 음성모델 준비는 업스트림 참고')

## 4. kr-custom-tts (학습형 — ESPnet, 자가 음성 학습)
본인 음성 데이터로 학습 → 모델 산출물을 받아 추론합니다.
**학습 세부 절차는 반드시 업스트림 가이드를 따르세요:** https://github.com/seastar105/kr-custom-tts

In [ ]:
%cd /content
![ -d kr-custom-tts ] || git clone https://github.com/seastar105/kr-custom-tts.git
%cd /content/kr-custom-tts
!ls -la   # 업스트림 구조 확인 (README/노트북/스크립트 위치)
# TODO(1): 의존성 설치 — 업스트림 README 의 설치 절차 그대로
#   !pip -q install espnet ...   (정확한 목록은 README/requirements 확인)
# TODO(2): 데이터 준비 — 본인 음성(wav)+전사(txt)를 업스트림이 요구하는 포맷으로
# TODO(3): 학습 실행 — 업스트림이 제공하는 학습 스크립트/노트북 셀 실행
# TODO(4): 학습된 모델 산출물 경로 확인 (예: exp/.../*.pth, *.onnx)
print('kr-custom-tts 클론 완료 — 학습 절차는 업스트림 가이드 참고')

## 5. SCE-TTS (학습형 — Tacotron2 + Vocoder)
Colab 가이드 기반의 자가 음성 커스터마이징.
**원 가이드를 따르세요:** https://gist.github.com/yunho0130 (SCE-TTS 가이드)

In [ ]:
%cd /content
# TODO(1): 원 가이드의 저장소/노트북을 클론하거나 가이드 셀을 그대로 사용
#   !git clone <SCE-TTS 가이드가 안내하는 저장소>
# TODO(2): 의존성 설치 (Tacotron2/vocoder) — 가이드 절차 그대로
# TODO(3): 데이터 준비 + 학습 실행 (가이드 셀)
# TODO(4): 학습된 Tacotron2 체크포인트(.pt) + vocoder 경로 확인
print('SCE-TTS — 원 Colab 가이드 절차를 따라 학습/내보내기 진행')

## 6. 모델 산출물 다운로드 → 로컬 연결
학습이 끝나면 산출물을 내려받아 로컬 저장소에 두고 `engines.py` 의 `model_path` 로 연결합니다.

In [ ]:
from google.colab import files
# 학습이 끝난 실제 산출물 경로로 바꿔서 내려받기 (예시 경로)
# files.download('/content/kr-custom-tts/exp/your_model.pth')
# files.download('/content/sce-tts/checkpoints/tacotron2.pt')
print('학습된 모델 파일 경로를 위 download() 인자로 지정해 내려받으세요.')

### 로컬에서 실측(real) 모드로 전환
1. 내려받은 모델을 로컬 저장소에 둡니다. 예: `models/kr_custom/your_model.pth`
2. `src/tts_bmt/engines.py` 에서 해당 엔진 생성 시 `model_path` 를 지정합니다.
   - 예: `KrCustomEngine(model_path='models/kr_custom/your_model.pth')`
   - (현재 `all_engines()` 는 인자 없이 생성하므로, 모델 경로를 넣어 주면 `available()` 이 True → real 모드)
3. `./run.sh` 실행 → `mode: real` 로 측정되고 `docs/` 가 실제 음질로 갱신됩니다.
4. `./deploy.sh` 로 Pages 에 배포.

> 추론형(Supertonic/piper-plus)은 Colab 없이 로컬에서 `./setup.sh --full` 로 설치해도 됩니다.
> 학습형(kr-custom-tts/SCE-TTS)만 이 Colab 학습이 필요합니다.